# Does simulated disagreement predict real controversy?

A backtest for [Lightningfish](https://github.com/rajul-kk/LightningFish) that scores something every previous run threw away, and that already broke once, which is why this notebook calibrates instead of guessing.

## Why this exists

Every earlier backtest reduced a finished simulation to `sign(final mean opinion)`, one bit, the same bit a single LLM call produces. At n=200:

| | accuracy |
|---|---|
| author-karma heuristic | 62.5% |
| **simulation** | **51.5%** |
| single LLM call | 50.0% |
| majority class | 50.0% |

p = 0.9994. The simulation is at chance, beating one raw call by 1.5 points.

The problem with calling the engine worthless from that: it evaluated a population on the one output where population has no structural advantage. A multi-agent run also produces a distribution, 24 opinions with a spread, and the mean says nothing about whether that crowd agreed. This notebook tests that: does the dispersion of simulated opinion predict whether an HN thread turned into an argument?

## What went wrong the first time

The first attempt (n=107) reported 53.3% and looked unremarkable until two problems surfaced. A real bug: `CachingAdapter` never got updated to delegate the new dispersion-scoring hook, so it silently scored the mean axis again, 53.3% never measured controversy at all. Fixed now, with a structural test enforcing it stays that way. And a guessed threshold: `stddev >= 0.35` was a bare guess, and the real range on that run was 0.085-0.286, never once reaching it. A threshold that never fires isn't a test.

This notebook calibrates the threshold from a held-out batch (median of its simulated stddevs) and reports accuracy only on a disjoint evaluation batch, so the threshold is chosen independently of what it's scored against (METHODOLOGY.md rules 3 and 6).

## Honest framing

This isn't a claim that a single call can't make the same prediction, `single_llm` does exactly that. The simulation derives disagreement from population heterogeneity rather than asserting it, real but narrower than "structurally impossible." Controversy prediction also isn't novel elsewhere (Reddit, Wikipedia edit wars). Given the mean axis is already at chance, expect this to fail too; the point is closing whether the standard evaluation measured the wrong output, not rescuing a result.


---
## What the data is

Source: the [Hacker News Algolia API](https://hn.algolia.com/api), free and unauthenticated (~10k req/hr), no key or scraping needed.

Sample: settled stories at least 24h old, pulled class-balanced. Each agent reads only submission-time fields, never the outcome:

| Field | Example |
|---|---|
| title | "China is now the world's greatest oil power" |
| author + karma | `bookofjoe` (110,566) |
| url domain | `economist.com` |
| type | story / Ask HN / Show HN |
| self-text | first 500 chars, if any |

Label: `num_comments / points` at settlement. 0.7+ is contested (a thread arguing with itself), under 0.4 is consensus (quietly upvoted), anything between is skipped as ambiguous. Stories under 20 points are skipped too: zero comments there means nobody saw it, not agreement.

That floor and gap remove most pulled stories, expect roughly one in three usable, hence the large `PULL_LIMIT`. Point-in-time safety is enforced in code: the seed enricher is a separate module from the ground-truth fetcher, and tests assert target values never leak into the seed.


---
## What the model is

Inference: [qwen2.5:7b](https://ollama.com/library/qwen2.5) (Q4_K_M, ~4.7 GB) served locally by Ollama. No API keys, $0. Fits a T4's 16 GB VRAM, the reason this runs here: the same job on a loaded CPU box managed zero completed events in 80 minutes.

The simulation runs 24 agents over 3-4 rounds, three tiers each round:

| Tier | Share | What happens |
|---|---|---|
| T1 originators | ~10% | LLM writes a structured post |
| T2 reactors | ~20% | LLM re-evaluates after reading a feed of others' posts |
| T3 drifters | rest | deterministic herding maths, no LLM call |

Agents are six HN archetypes with different resistance, recency bias, contrarian tendency, and herding coefficients. Heterogeneity is the point: identical agents would converge trivially and their spread would carry no information.

Scored: the standard deviation of the 24 final opinions, against a threshold calibrated by this run, not hardcoded. As with every backtest here, the simulation must beat every ladder rung:

| Rung | What it controls for |
|---|---|
| majority class | degenerate data |
| `naive` | Ask-HN / question-mark heuristic, no model |
| `single_llm` | one call asked the same controversy question |
| the simulation | (nothing, this is the thing being tested) |


---
## 1. Setup

Sidebar: **Accelerator → GPU**, **Internet → On**.

Install `zstd` before Ollama: its installer needs it to extract, Kaggle's image doesn't ship it, and skipping this makes the install fail quietly, surfacing later as a confusing `FileNotFoundError: 'ollama'`.


In [ ]:
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd > /dev/null 2>&1
!zstd --version || echo "WARNING: zstd missing - the install below will fail"
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import shutil, subprocess, time, requests

if shutil.which("ollama") is None:
    raise RuntimeError(
        "ollama not found after install. Scroll up for the installer's error: "
        "usually zstd (cell above) or Internet disabled in the sidebar."
    )

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("ollama up"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama installed but the server did not start")

In [ ]:
MODEL = "qwen2.5:7b"

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ollama pull {MODEL}

requests.post("http://localhost:11434/api/generate",
              json={"model": MODEL, "prompt": "hi", "stream": False, "keep_alive": -1},
              timeout=600)

for m in requests.get("http://localhost:11434/api/ps").json().get("models", []):
    vram = m.get("size_vram", 0) / 1e9
    print(f"{m['name']}: {vram:.2f} GB in VRAM")
    assert vram > 0, "model landed on CPU - enable the GPU accelerator, this is pointless otherwise"
print("GPU inference confirmed")

In [ ]:
!git clone --depth 1 https://github.com/rajul-kk/LightningFish.git /kaggle/working/lf
!pip -q install anthropic openai scipy requests pytest

import os, sys
os.chdir("/kaggle/working/lf")
sys.path.insert(0, "/kaggle/working/lf")

# Engine + HN suites only; also confirms the CachingAdapter delegation fix
# and the calibrated-threshold code are actually present in this clone.
!python -m pytest tests/core tests/hn -q 2>&1 | tail -3

---
## 2. Configuration

`PULL_LIMIT` needs headroom: the 20-point floor and mid-ratio gap discard ~2 of 3 pulled stories, then the calibration/evaluation split halves what's left. Aim for ~40 usable events per side.


In [ ]:
PULL_LIMIT = 300      # stories to pull; expect ~1 in 3 to be scoreable
N_AGENTS   = 24        # population size - the spread of THIS is what's scored
N_ROUNDS   = 4

os.environ["LIGHTNINGFISH_MODEL"] = f"ollama:{MODEL}"
os.environ["LIGHTNINGFISH_N_AGENTS"] = str(N_AGENTS)
os.environ["LIGHTNINGFISH_N_ROUNDS"] = str(N_ROUNDS)
os.environ["LIGHTNINGFISH_LOCAL_TIMEOUT"] = "120"
os.environ["PYTHONUNBUFFERED"] = "1"
print(f"{MODEL} | {N_AGENTS} agents x {N_ROUNDS} rounds | pulling {PULL_LIMIT}")

### Throughput check

Roughly 26 model calls per event. Worth confirming the per-call cost before committing to the full run. Double digits here means you're on CPU regardless of what the assertion above claimed.


In [ ]:
import time
from lightningfish_core.llm_provider import make_provider

provider = make_provider(f"ollama:{MODEL}")
t0 = time.time()
for _ in range(3):
    provider.get_opinion("Output ONLY a number between -1 and 1.", "Rate: 0.5", f"ollama:{MODEL}")
per_call = (time.time() - t0) / 3
print(f"{per_call:.2f}s per call  ->  ~{per_call*26:.0f}s per event")

---
## 3. Run it


In [ ]:
!python -m tests.integration.run_backtest hn-controversy-calibrated {PULL_LIMIT} 2>&1 | tee /kaggle/working/hn_controversy_calibrated.log


### Reading the log

Same shape as every calibrated-controversy run: how many events had a controversy direction, the calibration/evaluation split (either half under ~15 isn't enough to trust), and the derived threshold, worth comparing to the 0.35 that never fired last time. What matters is the final report: `beats_baselines` must PASS on every rung, `p_value_vs_best` well under 0.05. A warning about a constant predictor on the eval set means the two halves have different stddev distributions, or n is too small. Read that as inconclusive, not negative.


---
## 4. Save


In [ ]:
!cp -r .cache/lightningfish /kaggle/working/cache
!ls -la /kaggle/working/cache

The cache holds every simulated run's final distribution, so a later question about these same events (a different threshold, a different axis) costs nothing to re-score.

## Result interpretation

A negative here is expected and still worth recording. The mean axis is already at chance across n=200; if a properly calibrated dispersion axis also turns up nothing, that closes whether the standard evaluation measured the wrong output, cleanly this time, no bug or guessed threshold muddying it. Either way it belongs in [METHODOLOGY.md](https://github.com/rajul-kk/LightningFish/blob/main/METHODOLOGY.md).
